In [25]:
import os
import time
from dataclasses import dataclass, fields
from datetime import datetime

import flickr_api
import polars as pl
from dotenv import load_dotenv
from tqdm import tqdm

In [ ]:
load_dotenv()
flickr_api.set_keys(
    api_key=os.getenv("FLICKR_API_KEY"), api_secret=os.getenv("FLICKR_API_SECRET")
)

In [5]:
df = pl.read_csv("../dataset/csv/entities_need_more_scrap.csv")
df.head()

entity_id,wikimedia_url,poi_name,category,poi_name_tags
i64,str,str,str,str
6,"""https://commons.wikimedia.org/…","""Tsurugisaki Lighthouse""","""tower""","""tsurugisaki lighthouse,剱埼灯台,剣埼…"
16,"""https://commons.wikimedia.org/…","""Sendai Trust Tower""","""tower""","""sendai trust tower,仙台トラストタワー,s…"
45,"""https://commons.wikimedia.org/…","""Nagara Bridge, Osaka""","""bridge""","""長柄橋,長柄の人柱,nagara bridge, osaka"""
60,"""https://commons.wikimedia.org/…","""Shime Winding Tower""","""tower""","""志免鉱業所竪坑櫓,shime winding tower"""
108,"""https://commons.wikimedia.org/…","""Hayase Bridge""","""bridge""","""早瀬大橋,pont gran hayase,pont hay…"


In [19]:
@dataclass(kw_only=True)
class FlickrSpot:
    category: str = None
    id: str
    owner: str
    datetaken: int | None = None
    title: str = ""
    tags: str = ""
    latitude: str | None = None
    longitude: str | None = None
    place_id: str | None = None
    url_n: str | None = None
    img_id: str | None = None
    text: str | None = None

    def __post_init__(self):
        self.img_id = (
            f"{self.category}-".join(
                [
                    self.category,
                    str(self.latitude),
                    str(self.longitude),
                    str(self.datetaken),
                    self.owner,
                    self.id,
                    "n",
                ]
            )
            + ".jpg"
        )
        self.text = f"{self.title} {' '.join(self.tags)}".strip()

    @classmethod
    def from_photo(cls, photo: flickr_api.Photo, category: str) -> "FlickrSpot":
        data = photo.getInfo()
        location = data.get("location", None)
        if not location:
            return

        datetaken = int(
            datetime.strptime(data["taken"], "%Y-%m-%d %H:%M:%S").timestamp()
        )
        tags = data.get("tags", None)
        if tags:
            tags = " ".join([t.text for t in tags])

        lat = float(location["latitude"])
        lon = float(location["longitude"])

        return cls(
            category=category,
            id=photo.id,
            owner=data["owner"].id,
            title=data.get("title", ""),
            datetaken=datetaken,
            tags=tags,
            latitude=lat,
            longitude=lon,
            place_id=None,
            url_n=getattr(photo, "url_n", None),
        )

    def to_row(self) -> dict:
        """Dict in dataset/csv/spots/*.csv column order."""
        return {f.name: getattr(self, f.name) for f in fields(self)}

In [ ]:
for row in tqdm(df.to_dicts()):
    category = row["category"]

    csv_dir = f"./dataset/csv/additional_spots/{category}/"
    img_dir = f"./dataset/images/test_{category}/"
    os.makedirs(csv_dir, exist_ok=True)
    os.makedirs(img_dir, exist_ok=True)

    photos = []
    for p in flickr_api.Walker(flickr_api.Photo.search, tags=row["poi_name_tags"]):
        item = FlickrSpot.from_photo(p, category)
        if not item:
            continue

        photos.append(item.to_row())
        p.save(os.path.join(img_dir, f"{item.img_id}.jpg"))
        time.sleep(0.5)

    if photos:
        pl.DataFrame(photos).write_csv(os.path.join(csv_dir, f"{row['entity_id']}.csv"))

    time.sleep(1)

  0%|          | 2/533 [07:51<34:44:14, 235.51s/it]


HTTPError: HTTP Error 429: Too Many Requests

In [1]:
# recover deleted images by claude
import asyncio
import os

import httpx
import polars as pl
from tqdm.asyncio import tqdm

In [2]:
tower_df = pl.read_csv("../dataset/csv/additional_spots/tower_main.csv")
bridge_df = pl.read_csv("../dataset/csv/additional_spots/bridge_main.csv")
castle_df = pl.read_csv("../dataset/csv/additional_spots/palace_castle_main.csv")

In [7]:
semaphore = asyncio.Semaphore(8)
# IMG_DIR = "../dataset/images/{category}"


def write_file(path: str, data: bytes):
    with open(path, "wb") as f:
        f.write(data)


async def download(
    sem: asyncio.Semaphore,
    client: httpx.AsyncClient,
    img_id: str,
    url: str,
    category: str,
):
    if not url:
        return 
    
    target_dir = f"../dataset/images/{category}"
    os.makedirs(target_dir, exist_ok=True)

    async with sem:
        try:
            res = await client.get(url, timeout=60)
            res.raise_for_status()
        except httpx.HTTPError as e:
            print(f"Error fetching {url}. Reason: {e}")
            return

    await asyncio.to_thread(write_file, os.path.join(target_dir, img_id), res.content)


async with httpx.AsyncClient() as client:
    tasks = (
        [
            download(semaphore, client, row["img_id"], row["url_n"], category="tower")
            for row in tower_df.select(["img_id", "url_n"]).to_dicts()
        ]
        + [
            download(semaphore, client, row["img_id"], row["url_n"], category="bridge")
            for row in bridge_df.select(["img_id", "url_n"]).to_dicts()
        ]
        + [
            download(
                semaphore, client, row["img_id"], row["url_n"], category="palace_castle"
            )
            for row in castle_df.select(["img_id", "url_n"]).to_dicts()
        ]
    )

    await tqdm.gather(*tasks)

  7%|▋         | 4803/72359 [01:44<30:09, 37.33it/s]

Error fetching https://live.staticflickr.com/8008/7328654204_632af0ba00_n.jpg. Reason: Client error '404 Not Found' for url 'https://live.staticflickr.com/8008/7328654204_632af0ba00_n.jpg'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404


 16%|█▌        | 11749/72359 [04:57<32:29, 31.09it/s]

Error fetching https://live.staticflickr.com/4146/4844937367_775961c886_n.jpg. Reason: Client error '404 Not Found' for url 'https://live.staticflickr.com/4146/4844937367_775961c886_n.jpg'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404


 16%|█▋        | 11792/72359 [04:59<29:27, 34.27it/s]

...

 97%|█████████▋| 70357/72359 [30:24<00:28, 69.81it/s]

Error fetching https://live.staticflickr.com/1322/959551599_824b990c7f_n.jpg. Reason: Client error '404 Not Found' for url 'https://live.staticflickr.com/1322/959551599_824b990c7f_n.jpg'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404


100%|█████████▉| 72338/72359 [30:57<00:00, 95.44it/s] 

Error fetching https://live.staticflickr.com/7283/9412128262_5f7d051620_n.jpg. Reason: Client error '404 Not Found' for url 'https://live.staticflickr.com/7283/9412128262_5f7d051620_n.jpg'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
Error fetching https://live.staticflickr.com/2863/9409358843_272f40ea3a_n.jpg. Reason: Client error '404 Not Found' for url 'https://live.staticflickr.com/2863/9409358843_272f40ea3a_n.jpg'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404


100%|██████████| 72359/72359 [30:58<00:00, 38.94it/s]


In [17]:
tower_files = set(os.listdir("../dataset/images/tower"))
bridge_files = set(os.listdir("../dataset/images/bridge"))
castle_files = set(os.listdir("../dataset/images/palace_castle"))

In [21]:
tower_df = tower_df.with_columns(pl.col("img_id").is_in(tower_files).alias("is_recovered"))
bridge_df = bridge_df.with_columns(pl.col("img_id").is_in(bridge_files).alias("is_recovered"))
castle_df = castle_df.with_columns(pl.col("img_id").is_in(castle_files).alias("is_recovered"))

In [28]:
tower_df.filter(pl.col("is_recovered")).write_csv("../dataset/csv/additional_spots/tower_main_recovered.csv")
bridge_df.filter(pl.col("is_recovered")).write_csv("../dataset/csv/additional_spots/bridge_main_recovered.csv")
castle_df.filter(pl.col("is_recovered")).write_csv("../dataset/csv/additional_spots/palace_castle_main_recovered.csv")